# ⚡ PySpark Tricky Questions - Interactive Notebook
## Swiss Re Interview Prep

**Note**: These examples use conceptual code. Run on Databricks or local Spark for full execution.

In [1]:
# Setup - Run this first
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when, udf, broadcast, sum, avg, count
from pyspark.sql.types import StringType, IntegerType
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("InterviewPrep").master("local[*]").getOrCreate()
print("Spark initialized!")

Spark initialized!


---
## Q1: Transformations are LAZY

In [2]:
# NOTHING IS EXECUTED UNTIL AN ACTION!

df = spark.range(1000000)  # Not computed yet
filtered = df.filter(col("id") > 500000)  # Still not computed
doubled = filtered.withColumn("doubled", col("id") * 2)  # Still not!

print("No execution yet... all transformations are lazy.")

# THIS triggers execution:
result = doubled.count()  # ACTION!
print(f"Count (action triggered): {result}")

No execution yet... all transformations are lazy.
Count (action triggered): 499999


In [ ]:
# TRANSFORMATIONS (Lazy):
# select, filter, join, groupBy, withColumn, drop, distinct, repartition

# ACTIONS (Trigger execution):
# show, collect, count, write, take, first, head, reduce

---
## Q2: Narrow vs Wide Transformations

In [ ]:
df = spark.range(0, 10000).withColumn("key", col("id") % 10)

# NARROW (No shuffle - fast)
narrow_result = df.filter(col("id") > 5000).withColumn("new_col", col("id") * 2)

# WIDE (Shuffle required - slow)
wide_result = df.groupBy("key").agg(sum("id").alias("total"))

# See the plans:
print("--- Narrow (no Exchange in plan) ---")
narrow_result.explain()

print("\n--- Wide (look for 'Exchange' = shuffle) ---")
wide_result.explain()

In [ ]:
# NARROW: filter, select, map, withColumn, drop
# WIDE: groupBy, join, distinct, repartition, orderBy

# INTERVIEW TIP: "I always filter BEFORE groupBy to reduce shuffle size"

---
## Q3: UDF Performance (THE #1 TRAP!)

In [ ]:
# ❌ BAD: Python UDF (slow serialization)

@udf(StringType())
def slow_upper(s):
    return s.upper() if s else None

df = spark.createDataFrame([("hello",), ("world",)], ["text"])
result = df.withColumn("upper_text", slow_upper(col("text")))
result.show()

print("This works but is SLOW due to Python serialization")

In [ ]:
# ✅ GOOD: Use built-in functions!

from pyspark.sql.functions import upper

result = df.withColumn("upper_text", upper(col("text")))
result.show()

print("Built-in functions are 10-100x faster!")

In [ ]:
# ❌❌❌ WORST: API call inside UDF

# @udf(StringType())
# def get_hash_via_api(claim_id):
#     response = requests.get(f"https://api.com/hash/{claim_id}")
#     return response.json().get("hash")
#
# df.withColumn("hash", get_hash_via_api(col("claim_id")))

# WHY IT'S BAD:
# - 1 API call per ROW (1M rows = 1M calls = 27+ hours)
# - Spark retries failed tasks (duplicate calls)
# - Can hit rate limits

print("NEVER make API calls inside a UDF!")

---
## Q4: The mapPartitions Pattern

In [ ]:
# ✅ CORRECT: Use mapPartitions for external calls

import pandas as pd
from typing import Iterator

def process_partition(iterator: Iterator[pd.DataFrame]) -> Iterator[pd.DataFrame]:
    # Initialize connection ONCE per partition
    # session = requests.Session()  # Connection pooling
    
    for pdf in iterator:
        # Batch process all rows in this chunk
        pdf["hash"] = pdf["claim_id"].apply(lambda x: f"hash_{x}")
        yield pdf
    
    # Clean up
    # session.close()

# Apply:
df = spark.createDataFrame([("CL001",), ("CL002",)], ["claim_id"])
result = df.mapInPandas(process_partition, "claim_id string, hash string")
result.show()

print("Connection is reused for all rows in partition!")

---
## Q5: Broadcast Joins

In [ ]:
# Without broadcast: Sort-Merge Join (shuffles both tables)
# With broadcast: Broadcast Hash Join (no shuffle of large table)

large_df = spark.range(0, 100000).withColumn("key", (col("id") % 100).cast("string"))
small_df = spark.createDataFrame([(str(i), f"Label_{i}") for i in range(100)], ["key", "label"])

print("--- Without Broadcast (Sort Merge Join) ---")
result1 = large_df.join(small_df, "key")
result1.explain()

print("\n--- With Broadcast (Broadcast Hash Join) ---")
result2 = large_df.join(broadcast(small_df), "key")
result2.explain()

# Look for "BroadcastHashJoin" vs "SortMergeJoin"

---
## Q6: Skew Handling (Salting)

In [ ]:
from pyspark.sql.functions import rand, floor, expr, concat

# Simulate skewed data: 90% have key "A"
data = [("A", i) for i in range(9000)] + [("B", i) for i in range(1000)]
skewed_df = spark.createDataFrame(data, ["key", "value"])
dim_df = spark.createDataFrame([("A", "Info A"), ("B", "Info B")], ["key", "info"])

SALT_BUCKETS = 10

# Step 1: Salt the skewed (large) table
salted_large = skewed_df.withColumn(
    "salt", floor(rand() * SALT_BUCKETS).cast("int")
).withColumn(
    "salted_key", concat(col("key"), lit("_"), col("salt"))
)

# Step 2: Explode the small table
salted_small = dim_df.withColumn(
    "salt_array", expr(f"sequence(0, {SALT_BUCKETS - 1})")
).select(
    "key", "info",
    expr("explode(salt_array) as salt")
).withColumn(
    "salted_key", concat(col("key"), lit("_"), col("salt"))
)

# Step 3: Join on salted key
result = salted_large.join(salted_small, "salted_key")
print(f"Salted join result count: {result.count()}")
print("Skew is now distributed across multiple partitions!")

---
## Q7: Caching Gotchas

In [ ]:
df = spark.range(0, 100000)

# Caching is LAZY!
df.cache()  # This does NOT cache yet!

# First action triggers caching
import time
start = time.time()
count1 = df.count()  # Computes AND caches
print(f"First count: {time.time() - start:.3f}s")

# Second action uses cache
start = time.time()
count2 = df.count()  # Reads from cache
print(f"Second count: {time.time() - start:.3f}s")

# Clean up
df.unpersist()

---
## Q8: collect() Danger

In [ ]:
# ❌ DANGER: collect() brings ALL data to driver!

# big_df = spark.range(0, 10000000)
# all_data = big_df.collect()  # Can OOM the driver!

# ✅ SAFE: Use limit/take
df = spark.range(0, 10000000)
sample = df.limit(100).collect()  # Only 100 rows
print(f"Sample size: {len(sample)}")

# ✅ BETTER: Use write
# df.write.parquet("/path/to/output")  # No data to driver

---
## Q9: Partition Count

In [ ]:
df = spark.range(0, 10000)
print(f"Initial partitions: {df.rdd.getNumPartitions()}")

# repartition: Forces full shuffle
repartitioned = df.repartition(20)
print(f"After repartition(20): {repartitioned.rdd.getNumPartitions()}")

# coalesce: Reduces without shuffle (just merges)
coalesced = repartitioned.coalesce(5)
print(f"After coalesce(5): {coalesced.rdd.getNumPartitions()}")

print("\nRule: coalesce to reduce, repartition to increase")

---
## Q10: Window Functions

In [ ]:
from pyspark.sql.functions import row_number, rank, lag, lead

data = [
    ("A", "2024-01-01", 100),
    ("A", "2024-01-02", 150),
    ("A", "2024-01-03", 200),
    ("B", "2024-01-01", 80),
    ("B", "2024-01-02", 90),
]
df = spark.createDataFrame(data, ["region", "date", "sales"])

window = Window.partitionBy("region").orderBy("date")

result = df.withColumn("row_num", row_number().over(window)) \
           .withColumn("prev_sales", lag("sales", 1).over(window)) \
           .withColumn("running_total", sum("sales").over(
               Window.partitionBy("region").orderBy("date").rowsBetween(Window.unboundedPreceding, 0)
           ))

result.show()

---
## Summary Cheat Sheet

| Gotcha | Rule |
|--------|------|
| Lazy evaluation | Transformations don't execute until an action |
| Wide vs Narrow | Wide = shuffle, filter early |
| UDF performance | Use built-in functions, avoid Python UDFs |
| API in UDF | NEVER! Use mapPartitions |
| Small tables | Broadcast join them |
| Skewed data | Use salting or AQE |
| Caching | Cache is lazy, unpersist when done |
| collect() | Dangerous! Use limit/take or write |

In [ ]:
# Cleanup
spark.stop()